# 02 -- MetaPulsar (consistent) strategy on the real IPTA-DR2

The *consistent* combination strategy goes beyond a Frankenstein assembly by makign the timing models consistent across PTAs: every astrophysical parameter that lives in the merged components (`astrometry`, `spindown`, `binary`, `dispersion`) collapses to a **single fitted entry**, while detector-specific JUMPs / FD / DMX bins keep their per-PTA suffix. The result is a single `BasePulsar` whose timing model is astrophysically self-consistent.

1. **File and layout discovery** -- showcase MetaPulsar's regex-based directory walker, the canonical-name coordinate matcher, and `pta_summary`. These tools are how you go from "I have an IPTA release on disk" to "I have a `dict[pta_name -> list[par/tim entries]]` ready for `create_metapulsar`".
2. **Build a consistent MetaPulsar on real data** -- run the consistent combination on `J1853+1303` across EPTA dr2 + NANOGrav 9y, force a different reference PTA, and diff the rewritten consistent par files against the originals for manual check

## Step 0 -- Recover session state

We need `DATA_ROOT_STR` and `PULSAR_SUBSET` from `00_setup.ipynb`. If they are not in the IPython store, re-run `00_setup.ipynb`.

In [ ]:
import sys
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

import loguru

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# As before the hack because the PR is not merged yet
def quiet_loguru(level: str = "WARNING") -> None:
    loguru.logger.remove()
    loguru.logger.add(sys.stdout, level=level)

quiet_loguru()

%store -r DATA_ROOT_STR
%store -r PULSAR_SUBSET

DATA_ROOT = Path(DATA_ROOT_STR)
TARGET = PULSAR_SUBSET[0]
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"PULSAR_SUBSET  = {PULSAR_SUBSET}")
print(f"TARGET pulsar  = {TARGET}")

## Step 1 -- Layout discovery and `pta_summary`

MetaPulsar ships with a regex-based **layout discoverer** that scans a directory tree, identifies the conventions a PTA used for laying out its `.par`/`.tim` files (where they live, what suffixes they carry, which `INCLUDE` paths the tim files reference), and turns that into a `Layout` object. `combine_layouts` merges several `Layout`s into one, and `discover_files` resolves the actual files on disk.

Each PTA in IPTA-DR2 uses a different on-disk convention (EPTA's flat-per-pulsar tree, NANOGrav's release-versioned directories, PPTA's combined dr1+dr2 layout). The regex discoverer copes with all three out-of-the-box.

This discovery service has been tested with all currently publicly released PTA data (NANOGrav, PPTA, EPTA, MPTA, InPTA)

In [ ]:
from metapulsar import (
    combine_layouts,
    discover_files,
    discover_layout,
    filter_file_data_by_pulsars,
    get_pulsar_names_from_file_data,
    pta_summary,
)

# This works on all real PTA data releases we have tested with (EPTA, PPTA, NANOGrav, MPTA, InPTA...)
epta_layout = discover_layout(str(DATA_ROOT / "EPTA_v2.2"), name="EPTA dr2", verbose=False)
ppta_layout = discover_layout(str(DATA_ROOT / "PPTA_dr1dr2"), name="PPTA dr1dr2", verbose=False)
nanograv_layout = discover_layout(str(DATA_ROOT / "NANOGrav_9y"), name="NANOGrav 9y", verbose=False)

combined_layout = combine_layouts(epta_layout, ppta_layout, nanograv_layout)
file_data = discover_files(combined_layout, verbose=False)
quiet_loguru()

print("Discovered PTAs and pulsar counts:")
for pta, files in file_data.items():
    print(f"  {pta:<15s} -> {len(files):>3d} pulsars")

`get_pulsar_names_from_file_data` does **coordinate-based** matching across PTAs: it parses each `.par` file's RAJ/DECJ (or ELONG/ELAT), normalises to a canonical name, and returns the de-duplicated list of pulsars that appear in *any* PTA. This is what you want when one PTA labels a pulsar `J1853+1303` and another spells the J2000 name slightly differently or uses a `B`-name.

In [ ]:
pulsar_names = get_pulsar_names_from_file_data(file_data)
print(f"Coordinate-matched pulsars across all PTAs: {len(pulsar_names)}")
print("First few:", pulsar_names[:8])

`pta_summary` is the check you can do to see whether all pulsars have been discovered

In [ ]:
quiet_loguru()

pta_summary(file_data)

## Step 2 -- Filter to `PULSAR_SUBSET` and focus on `TARGET`

`filter_file_data_by_pulsars` shrinks `file_data` to just the pulsars in `PULSAR_SUBSET`. We then narrow further to a single pulsar (`TARGET = J1853+1303`) so the heavy `create_metapulsar` build stays under a minute on a laptop.

In [ ]:
# These are the pulsars currently in the subset
PULSAR_SUBSET

In [ ]:
filtered_data = filter_file_data_by_pulsars(file_data, PULSAR_SUBSET)

for pta, files in filtered_data.items():
    matched = sorted({Path(f["par"]).name for f in files})
    print(f"{pta:<15s} -> {matched}")

single_pulsar_data = {
    pta: [
        f
        for f in files
        if TARGET in Path(f["par"]).name or TARGET in Path(f["tim"]).name
    ]
    for pta, files in filtered_data.items()
}
single_pulsar_data = {pta: files for pta, files in single_pulsar_data.items() if files}
print(f"\nPTAs available for {TARGET}: {list(single_pulsar_data.keys())}")

## Step 3 -- Build a consistent MetaPulsar (auto reference PTA)

When `reference_pta=None` (the default), the factory picks the PTA with the longest timespan. We pre-stage the *original* per-PTA `.par` files into `./parfiles/` (so we can diff them against the rewritten consistent ones in step 5), then build the consistent MetaPulsar with `parfile_output_dir="./parfiles"`.

The bulk of the time is `parameter_manager.make_parfiles_consistent()` (parsing every par file into PINT, copying mergeable parameters, re-emitting new consistent par files). With only EPTA + NANOGrav for `J1853+1303` and ~1.5k TOAs, this is ~30-60 s on a laptop.

In [ ]:
import shutil

from metapulsar import create_metapulsar


# Back up the original par files
parfiles_dir = Path("./parfiles").resolve()
parfiles_dir.mkdir(exist_ok=True)

for pta, files in single_pulsar_data.items():
    src = Path(files[0]["par"])
    dest = parfiles_dir / f"{TARGET}_original_{pta}.par"
    shutil.copy(src, dest)
    print(f"  copied original  {pta:<15s} -> {dest.name}")

# Create the consistent MetaPulsar (output new par files with consistent timing models)
mp_consistent = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
    combine_components=["astrometry", "spindown", "binary", "dispersion"],
    add_dm_derivatives=True,
    parfile_output_dir=str(parfiles_dir),
)
quiet_loguru()

print()
print(f"Name              : {mp_consistent.name}")
print(f"Strategy          : {mp_consistent.combination_strategy}")
print(f"PTAs combined     : {list(mp_consistent._pulsars.keys())}")
print(f"Reference PTA     : {list(mp_consistent._pulsars.keys())[0]}")
print(f"Components merged : {mp_consistent.combine_components}")
print(f"Total TOAs        : {len(mp_consistent.toas)}")
print(f"Fit parameters    : {len(mp_consistent.fitpars)}")

## Step 4 -- Force a different reference PTA

The reference PTA contributes the *values* of every merged parameter -- it is therefore the only PTA whose `.par` is preserved verbatim. Forcing a different reference is mostly a sensitivity knob: a well-constrained pulsar should be statistically indistinguishable across reference choices. We rebuild the same pulsar with NANOGrav 9y as the reference (when present) and write to a separate `parfiles_ngref/` directory.

In [ ]:
# A rule to prefer NANOGrav 9y if it is present
FORCED_REF = (
    "NANOGrav 9y"
    if "NANOGrav 9y" in single_pulsar_data
    else next(iter(single_pulsar_data))
)

# Create the consistent MetaPulsar (output new par files with consistent timing models)
mp_consistent_ngref = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="consistent",
    reference_pta=FORCED_REF,
    parfile_output_dir="./parfiles_ngref",
)
quiet_loguru()

print(f"Forced reference PTA   : {FORCED_REF}")
print(f"PTAs (reference first) : {list(mp_consistent_ngref._pulsars.keys())}")
print(f"Fit parameter count    : {len(mp_consistent_ngref.fitpars)}")

## Step 5 -- Diff the original vs. consistent par files

For each non-reference PTA we side-by-side print the merged-component lines from the *original* par file (the one we copied from the IPTA-DR2 release) and from the *consistent* par file (rewritten by `parameter_manager.make_parfiles_consistent`). The values in the consistent file should now match the reference PTA's original values exactly; everything outside the merged components (JUMPs, EFAC/EQUAD per backend, DMX, ...) is left alone.

In [ ]:
import re

MERGED_PAR_KEYS = {
    "astrometry": ["RAJ", "DECJ", "PMRA", "PMDEC", "PX", "POSEPOCH", "ELONG", "ELAT"],
    "spindown": ["F0", "F1", "F2", "PEPOCH"],
    "binary": ["BINARY", "PB", "A1", "OM", "T0", "ECC", "EPS1", "EPS2", "TASC", "M2", "SINI"],
    "dispersion": ["DM", "DM1", "DM2", "DMEPOCH"],
}
FLAT_KEYS = sorted({k for ks in MERGED_PAR_KEYS.values() for k in ks})


def parfile_value_map(par_text: str) -> dict:
    out = {}
    for line in par_text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        tokens = re.split(r"\s+", line)
        key = tokens[0]
        if key in FLAT_KEYS and len(tokens) >= 2:
            out[key] = tokens[1]
    return out


ref_pta = list(mp_consistent._pulsars.keys())[0]
non_ref_ptas = [pta for pta in mp_consistent._pulsars.keys() if pta != ref_pta]

print(f"Reference PTA: {ref_pta}\n")
for pta in non_ref_ptas:
    orig = next(parfiles_dir.glob(f"*{TARGET}_original_{pta}.par"), None)
    cons = next(parfiles_dir.glob(f"*{TARGET}_consistent_{pta}.par"), None)
    if orig is None or cons is None:
        print(f"[skip] {pta}: missing original or consistent file (orig={orig}, cons={cons})")
        continue
    orig_map = parfile_value_map(orig.read_text())
    cons_map = parfile_value_map(cons.read_text())

    print(f"=== {pta}: merged-component keys, original -> consistent ===")
    keys = sorted(set(orig_map) | set(cons_map))
    for key in keys:
        o = orig_map.get(key, "--")
        c = cons_map.get(key, "--")
        marker = "" if o == c else "   <-- changed"
        print(f"  {key:<8s}  orig={o:<28s}  cons={c:<28s}{marker}")
    print()

## Step 6 -- Visualize the design matrix: composite vs. consistent

To make the *consistent* strategy concrete, build the same pulsar a second time with `combination_strategy="composite"` and `spy` the two design matrices side-by-side. Both `Mmat`s are reordered by PTA -- using the per-TOA `pta` flag MetaPulsar attaches to every TOA -- so the per-PTA blocks stack vertically, the same convention notebook 01 used for the FrankenPulsar.

* **Composite (left)** -- block-diagonal: each PTA's timing-model parameters only touch its own rows. This is the FrankenStat shape from notebook 01, reproduced here through MetaPulsar's `combination_strategy="composite"`.
* **Consistent (right)** -- the merged components (`astrometry`, `spindown`, `binary`, `dispersion`) collapse into single columns that span *all* rows, while detector-specific parameters (JUMPs, FD, DMX, ...) keep the block-diagonal structure.

In [ ]:
# Create a composite MetaPulsar (FrankenStat), equal to the consistent one above
mp_composite = create_metapulsar(
    file_data=single_pulsar_data,
    combination_strategy="composite",
)

In [ ]:
# Reorder fit parameters for the side-by-side spy plot below:
#   - drop the (many) DMX columns from the composite Mmat
#   - put shared (un-suffixed) parameters first, then group by PTA suffix.
# A parameter is per-PTA iff its name ends with `_<pta_name>` for one of the PTAs in
# `single_pulsar_data`. The composite case has no shared parameters, so this just
# groups by PTA. The consistent case naturally splits into a shared block followed
# by per-PTA detector parameters (JUMPs, FD, ...).
PTA_NAMES = list(single_pulsar_data)

def pta_group(name: str) -> int:
    """0 if `name` is shared, k+1 if it ends with `_<PTA_NAMES[k]>`."""
    for k, pta in enumerate(PTA_NAMES):
        if name.endswith(f"_{pta}"):
            return k + 1
    return 0


def reorder_cols(fitpars, keep=lambda n: True):
    """Indices of `fitpars` that pass `keep`, shared first then grouped by PTA."""
    return sorted(
        (i for i, n in enumerate(fitpars) if keep(n)),
        key=lambda i: (pta_group(fitpars[i]), i),
    )

# Column ordering for visual
comp_cols = reorder_cols(mp_composite.fitpars, keep=lambda n: not n.startswith("DMX"))
cons_cols = reorder_cols(mp_consistent.fitpars)

In [ ]:
# Sort TOAs by PTA so the per-PTA blocks stack vertically
# (same convention as the FrankenPulsar in 01_frankenstat_composite.ipynb)
isort_composite = np.argsort(mp_composite.flags["pta"], kind="stable")
isort_consistent = np.argsort(mp_consistent.flags["pta"], kind="stable")

fig, (ax_comp, ax_cons) = plt.subplots(1, 2, figsize=(11, 5), sharey=False)

ax_comp.spy(
    mp_composite.Mmat[np.ix_(isort_composite, comp_cols)] != 0,
    aspect="auto",
    markersize=1,
)
ax_comp.set_title(f"FrankenStat / composite  shape={mp_composite.Mmat.shape}")
ax_comp.set_xlabel(f"fit parameter index ({len(comp_cols)} shown, DMX hidden)")
ax_comp.set_ylabel("TOA index (sorted by PTA)")

ax_cons.spy(
    mp_consistent.Mmat[np.ix_(isort_consistent, cons_cols)] != 0,
    aspect="auto",
    markersize=1,
)
ax_cons.set_title(f"MetaPulsar / consistent  shape={mp_consistent.Mmat.shape}")
ax_cons.set_xlabel(
    f"fit parameter index ({len(cons_cols)} total: shared first, then per-PTA)"
)

fig.suptitle(f"Design matrix for {TARGET}: composite vs. consistent (rows sorted by PTA)")
fig.tight_layout()

## Step 7 -- Persist the file-data dict for notebook 03

MetaPulsar objects hold open file handles / thread locks and are therefore not picklable, which means we cannot pass them directly through IPython's `%store`. What we *can* persist is the plain `dict` of `{pta: [{par, tim, ...}, ...]}` entries that drives `create_metapulsar`. Notebook 03 picks that dict up and rebuilds the consistent MetaPulsar in a single call.

We deliberately do **not** rebuild the second pulsar (`B1953+29`) here. The recipe is identical to step 3; running it once for `J1853+1303` already exercises the full consistent path. For a production batch, see `create_all_metapulsars(file_data, combination_strategy="consistent")` in `examples/notebooks/using_metapulsar.ipynb`.

In [ ]:
FILE_DATA_REGISTRY = {TARGET: single_pulsar_data}
TUTORIAL_TARGET = TARGET
%store FILE_DATA_REGISTRY
%store TUTORIAL_TARGET
print("Stored FILE_DATA_REGISTRY and TUTORIAL_TARGET for 03_consistency_checks.ipynb.")
print("Notebook 03 will rebuild the consistent MetaPulsar from this dict.")